# Inference — final XGBoost pathogenicity classifier

Loads the **stage-2** deployable model (`models/xgb_final_pathogenicity.json` +
`*_meta.json` from `notebooks/train_final_xgboost.ipynb` / `python src/final_model.py`)
and scores **real ClinVar VUS** (uncertain significance) that were cleaned, joined
to UniProt, and position-matched separately from the pathogenic/benign training set.

Honest KPIs live in nested CV (`notebooks/cv_xgboost.ipynb`), not in this notebook.

Build the VUS table first (once):

```bash
python src/clean_clinvar.py
python src/join_clinvar_uniprot.py --vus
python src/position_matching_clinvar_uniprot.py --vus
```


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])
sys.path.insert(0, str(project_root / "src"))

from final_model import (
    DEFAULT_META_PATH,
    DEFAULT_MODEL_PATH,
    load_final_classifier,
    predict_pathogenicity,
)
from holdout_train_eval import (
    POSITION_MATCHED_VUS_IN,
    prepare_inference_frame,
)

INPUT_PARQUET = POSITION_MATCHED_VUS_IN
OUT_PARQUET = project_root / "data/processed/vus_inference_predictions.parquet"

## Load model

In [ ]:
model, meta = load_final_classifier(DEFAULT_MODEL_PATH, DEFAULT_META_PATH)
print(f"n_estimators={meta['n_estimators']}, threshold={meta['threshold']:.4f}")
print(f"trained on {meta['n_train']} variants / {meta['n_genes']} genes")
print("features:", meta["feature_columns"])
pd.Series(meta["params"], name="param").to_frame()

## Score ClinVar VUS

These variants were **never used as training labels**. Predictions are the model's
pathogenic-vs-benign call at the saved Youden threshold (not a ClinVar reclassification).

In [ ]:
raw = pd.read_parquet(INPUT_PARQUET)
frame = prepare_inference_frame(raw, labels={"vus"})
print(f"VUS to score: {len(frame):,} variants / {frame['gene'].nunique():,} genes")
print(frame["gene"].value_counts().head(10))

preds = predict_pathogenicity(frame, model=model, meta=meta)
print("\nPredicted labels:")
print(preds["pred_label"].value_counts())
print(f"\ny_proba: median={preds['y_proba'].median():.3f}, "
      f"mean={preds['y_proba'].mean():.3f}")
preds.sort_values("y_proba", ascending=False).head(15)

In [ ]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
preds.to_parquet(OUT_PARQUET, index=False)
print("Saved", OUT_PARQUET, preds.shape)